# GridCombat Autoresearch — Colab Edition

**Runtime required:** GPU — T4 (free tier) is sufficient for Qwen 2.5 3B at 4-bit

| Cell | Type   | Purpose |
|------|--------|---------|
| 1    | Python | Mount Drive, set path constants, export as env vars |
| 2    | Bash   | Install Node.js and Python packages |
| 3    | Bash   | Configure git identity |
| 4    | Bash   | Repo setup: restore from Drive bundle or initialise a new git repository |
| 5    | Bash   | First run only: copy JS files, make initial commit, and save bundle to Drive |
| 6    | Python | Load Qwen 2.5 3B Instruct at 4-bit |
| 7    | Python | Define all orchestrator functions (read before running 8) |
| 8    | Python | Run the experiment loop |

**Resuming after session expiry:** re-run cells 1, 2, 3, 4, 6, 7, 8. Skip cell 5.

## Cell 1 (Python) — Mount Drive and export paths

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_ROOT  = '/content/drive/MyDrive/gridcombat'
WORK_DIR    = '/content/gridcombat'
MODEL_CACHE = '/content/model_cache'  # local Colab disk (~80 GB free) -- model re-downloads each session
BUNDLE_PATH = '/content/drive/MyDrive/gridcombat/repo.bundle'

# Export so bash cells can use $DRIVE_ROOT, $WORK_DIR, etc.
os.environ['DRIVE_ROOT']  = DRIVE_ROOT
os.environ['WORK_DIR']    = WORK_DIR
os.environ['MODEL_CACHE'] = MODEL_CACHE
os.environ['BUNDLE_PATH'] = BUNDLE_PATH

os.makedirs(DRIVE_ROOT,  exist_ok=True)
os.makedirs(WORK_DIR,    exist_ok=True)
os.makedirs(MODEL_CACHE, exist_ok=True)

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Work dir    : {WORK_DIR}')
print(f'Model cache : {MODEL_CACHE}')
print(f'Bundle path : {BUNDLE_PATH}')
print('Drive mounted OK.')

Mounted at /content/drive
Drive root  : /content/drive/MyDrive/gridcombat
Work dir    : /content/gridcombat
Model cache : /content/model_cache
Bundle path : /content/drive/MyDrive/gridcombat/repo.bundle
Drive mounted OK.


## Cell 2 (Bash) — Install Node.js and Python packages

In [2]:
%%bash
echo '--- Node.js ---'
if ! command -v node &> /dev/null; then
    apt-get install -y nodejs 2>&1 | tail -3
fi
node --version

echo '--- Python packages ---'
pip install -q transformers accelerate bitsandbytes

echo 'Done.'

--- Node.js ---
v20.19.0
--- Python packages ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
Done.


## Cell 3 (Bash) — Configure git identity

In [3]:
%%bash
git config --global user.email 'autoresearch@colab.local'
git config --global user.name  'Autoresearch Bot'
git config --global init.defaultBranch main
echo 'Git identity:'
git config --global --list | grep user

Git identity:
user.email=autoresearch@colab.local
user.name=Autoresearch Bot


## Cell 4 (Bash) — Repo setup

Restores from Drive bundle if a previous session exists; otherwise initialises a new git repository.
Safe to re-run on session restart.

In [4]:
%%bash
mkdir -p "$WORK_DIR"
cd "$WORK_DIR"

if [ -f "$BUNDLE_PATH" ]; then
    echo 'Bundle found on Drive -- restoring repo...'
    if [ -d .git ]; then
        echo 'Repo already present, fetching latest from bundle.'
        git fetch "$BUNDLE_PATH" 'refs/heads/*:refs/heads/*'
    else
        git clone "$BUNDLE_PATH" .
    fi
    echo
    echo 'Recent history:'
    git log --oneline -5
    echo 'Repo restored.'
else
    if [ -d .git ]; then
        echo 'Repo already initialised.'
    else
        git init
        echo 'Fresh repo initialised.'
    fi
    echo 'Run Cell 5 to add game files (first run only).'
fi

Initialized empty Git repository in /content/gridcombat/.git/
Fresh repo initialised.
Run Cell 5 to add game files (first run only).


## Cell 5 (Bash) — Copy game files, initial commit, and save bundle

**First run only. Skip on resume.**

Upload `ai.js`, `baseline_ai.js`, `game_core.js`, `evaluate.js` via the Colab
file browser (left sidebar, upload icon) so they appear at `/content/`. Then run this cell.

**Fix applied:** After committing the JS files this cell now calls `git bundle create`
to persist the repo to Drive immediately. Without this step the initial commit would be
lost on session expiry because the working directory `/content/gridcombat` is ephemeral.

In [5]:
%%bash
UPLOAD_DIR='/content'
cd "$WORK_DIR"

ALL_OK=true
for f in ai.js baseline_ai.js game_core.js evaluate.js; do
    if [ -f "$UPLOAD_DIR/$f" ]; then
        cp "$UPLOAD_DIR/$f" "$WORK_DIR/$f"
        echo "Copied: $f"
    elif [ -f "$WORK_DIR/$f" ]; then
        echo "Already present: $f"
    else
        echo "MISSING: $f -- upload it then re-run this cell"
        ALL_OK=false
    fi
done

if [ "$ALL_OK" = true ]; then
    git add ai.js baseline_ai.js game_core.js evaluate.js
    git commit -m 'initial: game files'
    echo
    echo 'Saving bundle to Drive...'
    git bundle create "$BUNDLE_PATH" --all && echo "Bundle saved to $BUNDLE_PATH" || echo "ERROR: bundle save failed"
    echo
    echo 'Initial commit done. Proceed to Cell 6.'
fi

Copied: ai.js
Copied: baseline_ai.js
Copied: game_core.js
Copied: evaluate.js
[main (root-commit) fd46d7d] initial: game files
 4 files changed, 1348 insertions(+)
 create mode 100644 ai.js
 create mode 100644 baseline_ai.js
 create mode 100644 evaluate.js
 create mode 100644 game_core.js

Saving bundle to Drive...
Bundle saved to /content/drive/MyDrive/gridcombat/repo.bundle

Initial commit done. Proceed to Cell 6.


## Cell 6 (Python) — Load Qwen 2.5 3B Instruct

`Qwen/Qwen2.5-3B-Instruct` is a public, ungated model — no HuggingFace account
or token is required.

Downloads approximately 6 GB on first run to local Colab disk (`/content/model_cache`).
Download time is approximately 3-4 minutes on a new session. The model is not cached
to Drive and will re-download at the start of each session. 4-bit NF4 quantization
uses approximately 4-5 GB VRAM on a T4 (16 GB total), leaving sufficient headroom for
inference. The 7B model was found to consume 13.62 GB of the 14.56 GB available,
leaving insufficient memory for the inference buffer. The 3B model is used instead.
`PYTORCH_ALLOC_CONF=expandable_segments:True` is set to reduce memory
fragmentation and `MAX_NEW_TOKENS` is set to 4096, which provides sufficient
generating a complete `ai.js` file.

**Expected warning — safe to ignore:**
The `transformers`/`huggingface_hub` library attempts to read an `HF_TOKEN`
secret on startup regardless of whether the model requires one. Colab intercepts
this and may display a prompt asking to grant access, or print a warning such as
`UserWarning: Error while fetching HF_TOKEN secret value`. This is normal behaviour
from the library itself, not an error in the notebook. The download will proceed
correctly without a token.

In [6]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, cache_dir=MODEL_CACHE, trust_remote_code=True
)

print('Loading model at 4-bit NF4...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=MODEL_CACHE,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

print(f'Device : {next(model.parameters()).device}')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {used:.1f} GB used / {total:.1f} GB total')
print('Model ready.')

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device : cuda:0
VRAM   : 2.1 GB used / 15.6 GB total
Model ready.


## Cell 7 (Python) — Define orchestrator

**This cell must be run before Cell 8.** It defines all constants, file helpers,
git utilities, the evaluator, the inference function, and the prompt builder that
Cell 8 depends on. The experiment loop does not start until Cell 8 is run.

Note: Uses pure OS file I/O. Absolutely zero stdout capture, `subprocess`, or pipes.

In [7]:
import os, re, time
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
TEMPERATURE      = 0.4
MAX_EXP          = 0      # 0 = run forever; interrupt kernel to stop
EVAL_TIMEOUT_S   = 120
MAX_RETRIES      = 3
MAX_HISTORY_ROWS = 30
MAX_NEW_TOKENS   = 4096

AI_FILE       = 'ai.js'
BASELINE_FILE = 'baseline_ai.js'
RESULTS_FILE  = 'results.tsv'

REQUIRED_STRINGS = ['runAITurn', 'module.exports', 'shouldDefend']


# ── Logging ───────────────────────────────────────────────────────────────────
def log(msg):
    ts = datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}', flush=True)


# ── File helpers ──────────────────────────────────────────────────────────────
def read(filename):
    with open(os.path.join(WORK_DIR, filename), 'r', encoding='utf-8') as f:
        return f.read()

def write(filename, content):
    with open(os.path.join(WORK_DIR, filename), 'w', encoding='utf-8') as f:
        f.write(content)

def append_result(commit, win_rate, eval_time, status, description):
    wr   = f'{win_rate:.4f}'  if win_rate  is not None else '0.0000'
    et   = f'{eval_time:.1f}' if eval_time is not None else '0.0'
    desc = description.replace('\t', ' ')[:200]
    with open(os.path.join(WORK_DIR, RESULTS_FILE), 'a', encoding='utf-8') as f:
        f.write(f'{commit}\t{wr}\t{et}\t{status}\t{desc}\n')

def recent_results(n=MAX_HISTORY_ROWS):
    try:
        lines = read(RESULTS_FILE).strip().splitlines()
        header = lines[0] if lines else 'commit\twin_rate\teval_time_s\tstatus\tdescription'
        return '\n'.join([header] + lines[1:][-n:])
    except FileNotFoundError:
        return 'commit\twin_rate\teval_time_s\tstatus\tdescription'

def best_win_rate_from_history():
    best = 50.0
    try:
        for line in read(RESULTS_FILE).strip().splitlines()[1:]:
            parts = line.split('\t')
            if len(parts) >= 4 and parts[3].strip().lower() == 'keep':
                try:
                    wr = float(parts[1])
                    if wr > best: best = wr
                except ValueError:
                    pass
    except FileNotFoundError:
        pass
    return best


# ── Git helpers (Pure File I/O for info gathering) ────────────────────────────
def git_short_hash():
    # Reads directly from .git to avoid subprocess and capturing output entirely.
    try:
        head = read('.git/HEAD').strip()
        if head.startswith('ref: '):
            return read('.git/' + head[5:]).strip()[:7]
        return head[:7]
    except Exception:
        return 'unknown'

def git_commit(message):
    safe = message.replace('"', "'").replace('\n', ' ')[:120]
    os.system('git add ai.js')
    rc = os.system(f'git commit -m "{safe}"') >> 8
    return rc == 0

def git_reset_hard():
    os.system('git reset --hard HEAD~1')

def recent_git_log(n=10):
    # Parses the reflog file directly. No execution required.
    try:
        lines = read('.git/logs/HEAD').strip().splitlines()
        log_lines = []
        for line in reversed(lines[-n:]):
            parts = line.split('\t', 1)
            if len(parts) == 2:
                meta = parts[0].split()
                if len(meta) >= 2:
                    log_lines.append(f"{meta[1][:7]} {parts[1]}")
        return '\n'.join(log_lines) or '(no git history yet)'
    except Exception:
        return '(no git history yet)'

def save_bundle():
    rc = os.system(f'git bundle create "{BUNDLE_PATH}" --all') >> 8
    if rc == 0:
        log(f'  [drive] Bundle saved to {BUNDLE_PATH}')
    else:
        log('  [drive] Bundle save failed.')


# ── Evaluator (File-Backed Node Wrapper) ──────────────────────────────────────
def run_evaluator():
    # We dynamically create a JS runner that writes directly to disk from inside JS.
    # This entirely avoids Python stdout capturing, pipes, and shell redirection.
    js_wrapper = (
        "const fs = require('fs');\n"
        "const logFile = 'eval_out.log';\n"
        "fs.writeFileSync(logFile, '');\n"
        "const writeLog = (msg) => fs.appendFileSync(logFile, msg);\n"
        "process.stdout.write = writeLog;\n"
        "process.stderr.write = writeLog;\n"
        "console.log = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "console.error = (...args) => writeLog(args.join(' ') + '\\n');\n"
        "setTimeout(() => {\n"
        f"    writeLog('\\n[eval] TIMEOUT after {EVAL_TIMEOUT_S}s\\n');\n"
        "    process.exit(124);\n"
        f"}}, {EVAL_TIMEOUT_S} * 1000);\n"
        "try {\n"
        "    require('./evaluate.js');\n"
        "} catch(e) {\n"
        "    writeLog('\\n' + (e.stack || String(e)) + '\\n');\n"
        "    process.exit(1);\n"
        "}\n"
    )

    write('runner.js', js_wrapper)

    # Execute without redirection. All output lives securely inside eval_out.log.
    os.system('node runner.js')

    try:
        out = read('eval_out.log')
    except FileNotFoundError:
        out = ''

    wr = re.search(r'^win_rate:\s+([\d.]+)', out, re.MULTILINE)
    et = re.search(r'^eval_time_s:\s+([\d.]+)', out, re.MULTILINE)

    if not wr:
        log('  [eval] No win_rate in output. Tail:\n' + '\n'.join(out.splitlines()[-20:]))
        return None, None

    return float(wr.group(1)), float(et.group(1)) if et else 0.0


# ── Local inference ───────────────────────────────────────────────────────────
def call_local(prompt):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an expert game AI engineer. '
                'You reason carefully, make one focused change at a time, '
                'and always follow the output format exactly.'
            )
        },
        {'role': 'user', 'content': prompt},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


# ── Prompt ────────────────────────────────────────────────────────────────────
GAME_CONSTANTS_SUMMARY = """
## Game constants (read-only, from game_core.js)

Unit stats:  type         hp  move  capture  ranged  range
             infantry     10    3     yes      no      -
             mech         12    2     yes      no      -
             tank         10    2     no       no      -
             heavy        16    2     no       no      -
             artillery     8    2     no       yes    3-4
             rocket         7    2     no       yes    3-5

Damage table (attacker rows, defender cols):
             vs:  inf  tank  mech  heavy  arty  rocket
  infantry         5    2     3     2      4     3
  tank             8    6     5     4      5     6
  mech             6    5     5     3      6     5
  heavy           10    8     9     6      7     8
  artillery        9    8     8     6      5     7
  rocket           6   10     9     8      6     5

Terrain defense multiplier (lower = more damage taken):
  plain:0.85  wood:0.70  mountain:0.40  road:1.00  water:impassable

UNIT_VALUE: infantry:10  mech:30  tank:70  heavy:160  artillery:60  rocket:150

Combat: finalDamage = floor(baseDamage * (attacker.hp/maxHp) * terrainDef * (1-homeBonus))
Melee only: defender counter-attacks if alive.  Ranged: no counter.
Capture: capturer must stand on enemy HQ each turn; 2-10 capture points/turn.
Win: capture all enemy HQs, or eliminate all enemy units.
"""

def build_prompt(ai_js, results_history, experiment_num, best_wr):
    return f"""You are an autonomous AI researcher. Your job is to improve the game AI
in ai.js for a turn-based strategy game by modifying the heuristic decision logic.

## Current experiment: #{experiment_num}
## Best win_rate so far: {best_wr:.4f}  (baseline = 50.0, higher is better)
## Evaluation: 200 games (4 scenarios x 25 x both sides). Noise ~1.5 points.
{GAME_CONSTANTS_SUMMARY}

## Experiment history (results.tsv -- use this to avoid repeating failures):
{results_history}

## Recent git commits (last 10) -- do NOT reproduce any of these changes exactly:
{recent_git_log()}

## Current ai.js -- the ONLY file you may modify:
```javascript
{ai_js}
```

## Your task
Make ONE focused change to improve win_rate. Think about what has and has not
worked in the history above. Do not repeat a change that was already discarded.

Good targets:
- UNIT_THREAT_WEIGHT or UNIT_CAUTION values at the top
- hqPullWeight or approachWeight inside the movement sort
- shouldDefend() threshold logic
- calculateAttackValue() scoring terms
- Attack priority ordering (who fires first)
- New heuristics (retreat when hp < 30%, focus-fire, flanking bonus)
- Removing a heuristic that may be hurting performance

## Output format -- follow this EXACTLY (the parser is strict):
1. Return the complete modified ai.js inside a single ```javascript code block.
2. Do NOT include any text before the opening ```.
3. After the closing ```, write exactly one line starting with the word CHANGE:
   describing what you changed and your reasoning.

Example:
```javascript
<complete file here>
```
CHANGE: Raised hqPullWeight from 4 to 10 for non-capturers because artillery and
tanks were meandering instead of advancing toward the objective.
"""


# ── Response parsing ──────────────────────────────────────────────────────────
def extract_js(text):
    m = re.search(r'```javascript[^
]*
(.*?)```', text, re.DOTALL)
    if m: return m.group(1)
    m = re.search(r'```[^
]*
(.*?)```', text, re.DOTALL)
    if m: return m.group(1)
    stripped = text.strip()
    if stripped.startswith(("'use strict'", '"use strict"', '//', '/*')):
        return stripped
    return None

def extract_description(text):
    after_code = re.sub(r'```.*?```', '', text, flags=re.DOTALL).strip()
    m = re.search(r'^CHANGE:\s*(.+)', after_code, re.MULTILINE | re.IGNORECASE)
    if m: return m.group(1).strip()[:200]
    for line in after_code.splitlines():
        line = line.strip()
        if line and not line.startswith('`'):
            return line[:200]
    return 'no description provided'

def passes_sanity_check(code):
    for s in REQUIRED_STRINGS:
        if s not in code:
            return False, f'missing required string: {s}'
    if len(code) < 500:    return False, f'too short ({len(code)} chars)'
    if len(code) > 80_000: return False, f'too long ({len(code)} chars)'
    return True, 'ok'


print('Orchestrator defined. Run Cell 8 to start.')

Orchestrator defined. Run Cell 8 to start.


## Research Architecture — Two Complementary Loops

This notebook implements one of two intended research loops. They operate at
different levels and are designed to reinforce each other.

### Loop 1 — Autoresearch (this notebook)

Operates continuously and without human intervention. The model makes small,
focused changes to `ai.js` — primarily parameter adjustments and simple heuristic
additions — guided solely by win rate as a signal. It has no visibility into what
actually occurs during games. Its strengths are:

- Parametric optimisation of weights such as `hqPullWeight`, `UNIT_THREAT_WEIGHT`,
  and threat penalty multipliers
- Simple heuristic additions within the existing code structure
- Emergent behavioural improvements as a consequence of better-balanced parameters,
  without any explicit targeted fix being written

Its limitation is that it cannot diagnose systemic problems or produce substantial
algorithmic additions reliably, as win rate alone does not supply sufficient
information for that class of reasoning.

### Loop 2 — Directed Research (session-based, human-guided)

Operates through human observation and chat-based analysis. Known gameplay problems
— such as ranged units advancing past the front line, or units failing to coordinate
— are described in natural language and reasoned about with reference to the game
constants, damage tables, and code structure. This produces deliberate, targeted
changes that address specific diagnosed failures rather than searching blindly.

Its output is injected into `ai.js` manually before resuming the autoresearch loop,
which then refines the result further through parameter tuning.

### Division of Labour

| Class of change | Loop 1 | Loop 2 |
|---|---|---|
| Parameter tuning | Yes | No |
| Simple heuristic additions | Occasionally | Yes |
| Emergent behavioural correction | Yes | No |
| Targeted fix for a known problem | No | Yes |
| Substantial algorithmic additions | No | Yes |

### Injecting a Loop 2 change

1. Interrupt Cell 8 if running.
2. Edit `ai.js` in `/content/gridcombat/` directly with the targeted change.
3. Commit it manually: `git add ai.js && git commit -m 'directed: description'`
4. Save the bundle: `git bundle create "$BUNDLE_PATH" --all`
5. Resume Cell 8. The autoresearch loop will continue from the new baseline.

## Cell 8 (Python) — Run the experiment loop

Stop at any time with **Runtime > Interrupt execution**.
Bundle is saved to Drive on every KEEP so progress survives session expiry.
On session restart re-run cells 1, 2, 3, 4, 6, 7, then this cell.

In [ ]:
import os
os.chdir(WORK_DIR)

for f in [AI_FILE, BASELINE_FILE]:
    if not os.path.exists(os.path.join(WORK_DIR, f)):
        raise FileNotFoundError(f'{f} not found in {WORK_DIR}. Run Cell 5 first.')

if not os.path.exists(os.path.join(WORK_DIR, RESULTS_FILE)):
    write(RESULTS_FILE, 'commit\twin_rate\teval_time_s\tstatus\tdescription\n')

log(f'Model           : Qwen2.5-3B-Instruct (local, 4-bit NF4)')
log(f'Temperature     : {TEMPERATURE}')
log(f'Max experiments : {"inf" if MAX_EXP == 0 else MAX_EXP}')
log(f'Eval timeout    : {EVAL_TIMEOUT_S}s')
log(f'Work dir        : {WORK_DIR}')
log(f'Drive bundle    : {BUNDLE_PATH}')
log('')

best_wr              = best_win_rate_from_history()
experiment_num       = 0
consecutive_failures = 0
log(f'Best win_rate from history: {best_wr:.4f}')

while True:
    experiment_num += 1
    if MAX_EXP > 0 and experiment_num > MAX_EXP:
        log(f'Reached MAX_EXPERIMENTS={MAX_EXP}. Stopping.')
        break

    log('')
    log('=' * 60)
    log(f'Experiment #{experiment_num}  |  Best: {best_wr:.4f}')
    log('=' * 60)

    # ---- Inference ----
    ai_js  = read(AI_FILE)
    prompt = build_prompt(ai_js, recent_results(), experiment_num, best_wr)
    log('Running local inference...')
    try:
        response_text        = call_local(prompt)
        consecutive_failures = 0
    except Exception as e:
        log(f'Inference error: {e}')
        consecutive_failures += 1
        if consecutive_failures >= MAX_RETRIES:
            log('Too many consecutive failures. Stopping.')
            break
        experiment_num -= 1
        time.sleep(2)
        continue

    # ---- Parse ----
    new_code    = extract_js(response_text)
    description = extract_description(response_text)

    if new_code is None:
        log('Could not extract JS. Skipping.')
        log('Preview: ' + response_text[:400].replace('\n', ' '))
        experiment_num -= 1
        continue

    ok, reason = passes_sanity_check(new_code)
    if not ok:
        log(f'Sanity check failed: {reason}. Skipping.')
        experiment_num -= 1
        continue

    log(f'Proposed: {description}')

    # ---- Commit ----
    write(AI_FILE, new_code)
    if not git_commit(description):
        log('git commit failed (nothing changed). Restoring.')
        os.system('git checkout HEAD -- ai.js')
        experiment_num -= 1
        continue

    commit_hash = git_short_hash()
    log(f'Committed {commit_hash}')

    # ---- Evaluate ----
    log('Running evaluator...')
    win_rate, eval_time = run_evaluator()

    if win_rate is None:
        log('CRASH -- evaluator returned no win_rate. Reverting.')
        git_reset_hard()
        append_result(commit_hash, None, None, 'crash', description)
        continue

    delta = win_rate - best_wr
    log(f'win_rate: {win_rate:.4f}  ({delta:+.4f} vs best)  eval_time: {eval_time:.1f}s')

    # ---- Keep or discard ----
    if win_rate > best_wr:
        best_wr = win_rate
        log(f'KEEP -- new best: {best_wr:.4f}')
        append_result(commit_hash, win_rate, eval_time, 'keep', description)
        save_bundle()
    else:
        git_reset_hard()
        log('DISCARD -- reverted to previous best.')
        append_result(commit_hash, win_rate, eval_time, 'discard', description)

[07:37:46] Model           : Qwen2.5-3B-Instruct (local, 4-bit NF4)
[07:37:46] Temperature     : 0.4
[07:37:46] Max experiments : inf
[07:37:46] Eval timeout    : 120s
[07:37:46] Work dir        : /content/gridcombat
[07:37:46] Drive bundle    : /content/drive/MyDrive/gridcombat/repo.bundle
[07:37:46] 
[07:37:46] Best win_rate from history: 50.0000
[07:37:46] 
[07:37:46] ============================================================
[07:37:46] Experiment #1  |  Best: 50.0000
[07:37:46] ============================================================
[07:37:46] Running local inference...
[07:42:57] Proposed: Improved the movement sorting by adding a new heuristic to prioritize capturing structures over attacking units. This reduces the tendency of units to wander around unnecessarily.
[07:42:57] Committed 1794e9c
[07:42:57] Running evaluator...
[07:42:57]   [eval] No win_rate in output. Tail:
Evaluating ai.js vs baseline_ai.js — 200 games

ReferenceError: shouldDefend is not defined
    at ru

## Architecture Note — Two-Loop AI Development Strategy

### Why the 3B Model is the Correct Choice for the Autoresearch Loop

Inference time per experiment is as significant a constraint as VRAM. Even if a
larger runtime such as Colab+ provided sufficient VRAM to load a 7B model without
error, inference time per experiment would scale proportionally — potentially
exceeding 10 minutes per call rather than approximately 4 minutes for the 3B.
Over hundreds of experiments that difference compounds into days of lost throughput.

The win rate signal carries approximately 1.5 points of noise across 200 games.
A model that proposes marginally better changes but requires three times the
inference time will produce fewer experiments per session and therefore less useful
signal overall. Fast iteration is more valuable than marginally better proposals.

The 3B model is therefore not a compromise forced by hardware constraints — it is
the correct instrument for this loop. The autoresearch loop is a search problem,
and search problems are governed by iteration rate as much as by the quality of
any individual step. A slower model does not compensate for its speed disadvantage
through better reasoning at this scale of problem and this level of signal resolution.

### The Two-Loop Strategy

The autoresearch loop above operates within a narrow band of possible improvements.
It adjusts parameters and simple heuristics — weights, thresholds, priority orderings
— guided solely by the win rate signal. It has no visibility into what is actually
happening during games: which units are dying, where, and under what conditions.
Improvements in this loop are real but limited in scope. Beneficial emergent effects
are possible — for example, rebalancing movement weights may incidentally correct
artillery units advancing ahead of melee units — but deliberate algorithmic reasoning
is beyond its reach.

A second, complementary loop addresses this limitation. It is human-guided and
driven by observation: gameplay descriptions, chat session analysis, and strategic
reasoning about specific failure modes. Where the autoresearch loop searches blindly,
the human-guided loop diagnoses deliberately and produces targeted algorithmic
additions — formation logic, coordinated unit behaviour, positional awareness rules
— that the autoresearch loop cannot discover on its own.

The two loops are not redundant. They operate at different levels and are intended
to be used in combination:

1. The human-guided loop identifies a structural problem, reasons about a solution,
   and introduces new algorithmic capability into `ai.js`.
2. The autoresearch loop then fine-tunes the parameters within that expanded
   capability, optimising what the human-guided loop added.

Richer game state context — unit positions, movement sequences, engagement outcomes
logged per game — would improve both loops. It would give the human-guided loop
concrete evidence to reason from, and could in principle be fed into the autoresearch
prompt to narrow the model's search toward productive changes.

A larger language model would also improve the autoresearch loop's algorithmic
reasoning, but model capacity and context quality are independent constraints that
must both be addressed. A larger model without richer context remains unable to
diagnose systemic problems; richer context without sufficient model capacity produces
plausible but unreliable code. The two factors compound each other.

### Research Strategy and Validation Framework

**Computational constraints and session design:**
Each Colab session yields approximately 60 experiments at the current inference rate
of approximately 12 minutes per experiment. Sessions require manual restart and model
re-download. This is not a platform for sustained convergence over weeks — it is a
tool for capturing large, obvious improvements in a limited number of deliberate
sessions. Blind continuation across many sessions is wasteful. Each session must be
preceded by a formal hypothesis derived from prior results, with a defined success
criterion. If the results do not confirm the hypothesis, that is the finding. This
follows the same discipline as a research proposal — resources are finite, time is
irreplaceable, and unfocused experimentation produces noise rather than knowledge.

**The two measures of success:**
The win rate measures AI versus AI performance against a fixed baseline opponent.
This is a necessary but not sufficient measure of improvement. The true and final
measure is whether the improvements make the AI meaningfully harder for a human
opponent to defeat. These are independent questions. An AI can improve its win rate
against a fixed baseline while remaining trivially exploitable by a human who plays
differently. Both must be tested separately.

**Human play testing:**
Validation against a human opponent must be conducted across multiple maps and under
varied strategies known only to the human player. The AI has no knowledge of the
human's approach and cannot adapt to a specific opponent. If the AI holds up against
novel strategies it was never evaluated against, that is meaningful validation. If it
collapses, that defines the next hypothesis. This testing can and should be conducted
with limited experimental results — it does not require convergence to be informative.

**The practical criterion:**
If AI versus AI improvements do not translate to harder human play within our
observable constraints, the procedure has no practical value for this project.
We are not generating academic results. The sole criterion of success is whether
the game becomes harder to defeat for a human player. That determination requires
direct observation, not inference from win rate alone.

**On the search space:**
Large improvements appear early and diminish as the search matures. The 11.75 point
gain from the first successful experiment illustrates this — obvious parameter
changes produce large effects initially. Subsequent sessions should be designed
around specific hypotheses about remaining productive regions of the search space,
not continued blind exploration. The question before each session is: what specific
improvement do we predict exists and why, and what result would refute that prediction.

## Findings, Conclusions, and Future Directions

### What Was Established

This notebook constitutes a formal system for autonomous game AI research within
severe hardware constraints. It is not a prototype — it is a complete, documented,
and reproducible research instrument. The following was established empirically:

The autoresearch loop functions correctly. The harness handles inference failures,
syntax errors, missing functions, and session interruption gracefully and without
human intervention. A win rate improvement of 11.75 points was achieved in the
first successful experiment, confirming the pipeline produces real results.

The Qwen 2.5 3B model at 4-bit NF4 quantization is the correct instrument for
this loop on a T4 GPU. Larger models exceed available VRAM or inference time
budgets. The 3B fits and runs at approximately 12 minutes per experiment.

### What Was Refuted

The 3B model is an insufficient agent for sustained autonomous code modification.
It produces valid output under ideal conditions but fails consistently across
iterations — repeating proposals, producing syntax errors, and omitting required
functions. This is a known boundary condition of small models in agentic use,
not a harness deficiency. The harness compensates for failures but cannot
compensate for a model that cannot genuinely vary its proposals.

The technical cause is well established. Models below approximately 7B parameters
lose syntactic coherence under long-form code generation. The model maintains
structural integrity in the early portion of a file and degrades as the context
fills — producing a consistent crash at a predictable location rather than a
random one. This was observed empirically: four consecutive crashes at line 91
with identical proposals, confirming the model is fixating rather than exploring
and degrading at the same point in every generation. This is not a prompt problem
or a configuration problem. It is a capacity problem.

A code-specific model such as Qwen 2.5 Coder would address this, but its minimum
viable size for reliable long-form generation — approximately 7B — exceeds the
T4 VRAM budget. The constraint is therefore architectural and not resolvable
within the current hardware without a fundamental change in platform.

Colab free tier is insufficient as a sustained research platform. Session limits,
manual restart requirements, and the absence of background execution constrain
the practical experiment budget to dozens of experiments per day of attentive
human operation. This is adequate for capturing large early improvements but
insufficient for convergence.

### The Boundary Condition

The autoresearch loop is a tool for capturing large, obvious improvements through
parameter search. It is not a tool for discovering novel algorithms. That boundary
is determined by the model capacity, the signal noise of the evaluator, and the
session duration of the platform. All three factors converge on the same conclusion:
the loop is productive in its early phase and diminishing thereafter.

### The Unresolved Question

Whether AI versus AI win rate improvements translate to harder human play remains
untested. This is the sole criterion of practical success and must be evaluated
through direct play testing across multiple maps and strategies before any
conclusion on the value of the approach can be drawn.

### Future Directions

Three directions are identified, in order of tractability within current constraints:

1. **Human play testing.** Test the current best AI against a human opponent across
   multiple maps and varied strategies. This requires no additional compute and
   answers the most important open question directly.

2. **Spatial awareness model for ranged units.** Design a general pairwise distance
   table for artillery and rocket units with a small set of tunable parameters.
   This is a single algorithmic addition that the autoresearch loop can then
   parameterise. It addresses the most significant exploitable gap between the
   current AI and human-level play — spatial positional awareness — through an
   empirical rather than hand-crafted approach. The parameters would be too numerous
   and subtle for a human to perceive during play, producing an AI that appears
   anticipatory. This requires human implementation or a capable model in the
   human-guided loop before the autoresearch loop can explore it.

3. **A more capable small model.** As the Qwen family and others advance, a future
   3B or 4B model with stronger instruction-following and code generation capability
   may resolve the agent reliability problem without exceeding the T4 constraints.
   This requires no architectural change — only a model ID update in Cell 6.

### A Note on the Contribution

This notebook is the technical report. It documents not only the system but the
reasoning behind every decision, the constraints that shaped the architecture, the
failures encountered and their causes, and the boundary conditions of the approach.
That documentation is the primary contribution. It saves subsequent researchers
the weeks of effort required to rediscover these boundaries independently and
provides a reproducible foundation from which the next advance can begin.